# Sentence splitting

Splits every raw document into sentence-level rows without touching anything in `data/raw`. Outputs go to `data/sentences/`:

- `everlane_web_sentences.csv`
- `gdelt_articles_sentences.csv`
- `reddit_posts_sentences.csv` (title and body kept separate via a `part` column)
- `reddit_comments_sentences.csv`

Reddit posts that link out to images/galleries/external articles don't have a recoverable `post_id` from their `url` (it only points at the external link, not the reddit permalink) — those rows get `post_id = NaN` and just won't join to `reddit_comments_sentences.csv`.

In [ ]:
import sys
!{sys.executable} -m pip install nltk pandas

In [ ]:
import os
import re

import pandas as pd
import nltk

nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

RAW_DIR = "../data/raw"
OUT_DIR = "../data/sentences"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def split_into_sentences(text):
    """Line-aware sentence split: paragraph/header breaks are respected so an
    unpunctuated line (e.g. a marketing header or a short reddit line) doesn't
    get glued onto the next line's first sentence."""
    if pd.isna(text):
        return []
    sentences = []
    for line in str(text).split("\n"):
        line = line.strip()
        if not line:
            continue
        sentences.extend(s.strip() for s in sent_tokenize(line) if s.strip())
    return sentences

## Everlane web pages

In [ ]:
df_web = pd.read_csv(f"{RAW_DIR}/everlane_web.csv")

rows = []
for _, row in df_web.iterrows():
    for i, sent in enumerate(split_into_sentences(row["text"])):
        rows.append({
            "source": row["source"],
            "page": row["page"],
            "url": row["url"],
            "date": row["date"],
            "sentence_id": i,
            "sentence": sent,
        })

df_web_sentences = pd.DataFrame(rows)
df_web_sentences.to_csv(f"{OUT_DIR}/everlane_web_sentences.csv", index=False)
print(f"{len(df_web)} pages -> {len(df_web_sentences)} sentences")
df_web_sentences.head()

## GDELT news articles

In [ ]:
df_gdelt = pd.read_csv(f"{RAW_DIR}/gdelt_articles_usable.csv")

rows = []
for _, row in df_gdelt.iterrows():
    for i, sent in enumerate(split_into_sentences(row["text"])):
        rows.append({
            "url": row["url"],
            "title": row["title"],
            "publish_date": row["publish_date"],
            "sentence_id": i,
            "sentence": sent,
        })

df_gdelt_sentences = pd.DataFrame(rows)
df_gdelt_sentences.to_csv(f"{OUT_DIR}/gdelt_articles_sentences.csv", index=False)
print(f"{len(df_gdelt)} articles -> {len(df_gdelt_sentences)} sentences")
df_gdelt_sentences.head()

## Reddit posts (title + body kept separate)

In [ ]:
df_posts = pd.read_csv(f"{RAW_DIR}/reddit_posts.csv")
df_posts["post_id"] = df_posts["url"].str.extract(r"/comments/([a-z0-9]+)/")

rows = []
for _, row in df_posts.iterrows():
    sentence_id = 0
    for sent in split_into_sentences(row["title"]):
        rows.append({
            "post_id": row["post_id"],
            "subreddit": row["subreddit"],
            "date": row["date"],
            "score": row["score"],
            "url": row["url"],
            "part": "title",
            "sentence_id": sentence_id,
            "sentence": sent,
        })
        sentence_id += 1
    for sent in split_into_sentences(row["text"]):
        rows.append({
            "post_id": row["post_id"],
            "subreddit": row["subreddit"],
            "date": row["date"],
            "score": row["score"],
            "url": row["url"],
            "part": "body",
            "sentence_id": sentence_id,
            "sentence": sent,
        })
        sentence_id += 1

df_posts_sentences = pd.DataFrame(rows)
df_posts_sentences.to_csv(f"{OUT_DIR}/reddit_posts_sentences.csv", index=False)
print(f"{len(df_posts)} posts -> {len(df_posts_sentences)} sentences "
      f"({df_posts['post_id'].isna().sum()} posts had no recoverable post_id)")
df_posts_sentences.head()

## Reddit comments

In [ ]:
df_comments = pd.read_csv(f"{RAW_DIR}/reddit_comments.csv")
df_comments["comment_id"] = df_comments.index

rows = []
for _, row in df_comments.iterrows():
    for i, sent in enumerate(split_into_sentences(row["text"])):
        rows.append({
            "post_id": row["post_id"],
            "comment_id": row["comment_id"],
            "subreddit": row["subreddit"],
            "date": row["date"],
            "score": row["score"],
            "sentence_id": i,
            "sentence": sent,
        })

df_comments_sentences = pd.DataFrame(rows)
df_comments_sentences.to_csv(f"{OUT_DIR}/reddit_comments_sentences.csv", index=False)
print(f"{len(df_comments)} comments -> {len(df_comments_sentences)} sentences")
df_comments_sentences.head()